# RepairScope-X — Notebook 30: Evidence Integration and Independent NB29 Audit

**CPU-only; no paid API and no GPU.** Original Notebook 27's **710** reconciled repair-trial records and Notebook 21's **64** completed MDLM repair trials are embedded byte-for-byte and independently SHA-256-verified.

Optionally supply the output ZIP from **Notebook 29B**. This audit separates the 774 historical records from the new model-specific semantic-repair experiment and never treats repeated seeds or shared physical proposals as independent problems.

**What this notebook does:** validate source bytes and trial identifiers, generate eight-cohort historical tables, inspect NB29 for missing stages and content-hash mismatches, calculate held-out distinct-problem recovery and paired exact sign tests, report stage-level decoder cost without assuming matched budgets, and export manuscript CSVs, plots, a claim-gating report, and a verified ZIP.

**It cannot replace model inference or genuine external validation.** If the NB29 ZIP is absent or partial, it publishes only historical descriptive statistics, explicitly marking the new experiment incomplete.


### 1. Environment and embedded input hashes

In [ ]:
import os, io, json, re, math, hashlib, zipfile, zlib, base64, csv, statistics, warnings
from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
OUT=Path('/content/RepairScopeX_NB30_Audit_Outputs') if Path('/content').exists() else Path.cwd()/'RepairScopeX_NB30_Audit_Outputs'
OUT.mkdir(parents=True,exist_ok=True)
print('CPU audit output directory:',OUT)

### 2. Frozen historical evidence: Notebook 27 + completed Notebook 21

In [ ]:
EMBEDDED = {'nb27_csv': '...', 'nb21_jsonl': 'eNrtnflv49iRx3/fv4Lo7GJnMj3Ku4/G9gLZyQ6SIJkEmckOFvHArZbpttCy5NXRTmOS/32r6pESJVEyqcuyXe72IfKx3lEU66N3fN/Pr/pXr95kr3qj2XB6Nbofvul1B/334+60Pxq+kUHYN0b8Y9rtD5T9hw4yqlevs1d349H7QX572eBaTH7VnXYn+XQp7eX1OJ/c4NluD9PjyZQNHpvkOZqmDOHlaNz/0B92B5ej2fRuRoZ+uMmz4ez2fT6eZN1xnqn4OoP/3eFVFrL872B08DkbDXt5Nh1lUN6rGf4JF0274w/5NJOqk/0ln87Gw6ybfYJyX4GZ/vTmNp/2e2DgDoo3gXKRxRy+7+Fk9u7b33336z9c/vq773/877+8yf4jleA/33UuhnNjPci1P8nhwsl9Pu5k347G2f1ofJUVrTbJrvvD/uQmWaw3mK76pmys7EP/U/5AOWeT/vBDlnd7N9lkdnc36OdXRQNRM0AJL4bYaLvWVcXsqwy/Q/YWWu/d3OByfbP+JHuHp9GN4/yu2x/nV+y3x/Nb6Yv5e2g6+pgPL+mNCA6RwsBJyPBynI/u8iG975Qrjg1GvY90xAd62+eTfPypW7xfRcfHILyLSqefEtKkK1Imk0tyKCSdjmc5nOzddIcfyrOXd6NJH01NMEe89n48Gn64TO0PxenNxuMcmuASSpKP+72FHTJ7mZKnW6uSHJJddwcTTAePnY+Xk1mvB+1UOTyZgrVpzYkio8qZIsNPcPi63ytr/vPhLBdXQNNe9csH4XA2GFSuWToH97/AAuHdcLm4BYqr/omXQSk/5Zdwk9yM6Pn8x9/84Y/fd2/vBnBX94fX/cEgvTf/b5ZPpuAN+HFHPlB0K1zn5R1w3x0MLic5vDGu8LzuaO+MMspZAV/KYpq7vPvx8tO4e3v54T1e19HOaCO9tkHHYDTeN+Pu/eX8UZDuDEj6Nx/s68xKAW9/q+BPpaBm0sMfUA4piwPlC610eQT+cCFC7lpYuE7jaYgUeN6DMSVdLK25ZFHCOSgV2nQGf0SPB3yAF97glWQ+aDBnDJRZWgd/xhDwhfPeQhjSUVu0gFk6MGxEgKP4AJPBaayFjaHIwWkb4ZdR2JCYu0wmDNVWK/jbKbRNuTkXbXwgH2mUd6UdyC84zNuFzVWSzmHVAzWAV/gWk9qrebOhVfLB3s2x1NYq2IA5WukMnJMxFC/0Y/3+Cd9n/Q/wxpiN8eZ+dWW8v7ZXzl0LLfMcbuaesOIqXlmj7bVQTl4p+MvGrlBBw418fe3zXrebe3113dOeEGUK5vA+fjX6+Oqf//JzW5DSpwYpzSDFIMV+Y5BikHp0kIK4HaJxxhFIGeYo5ijmqJYcZcVJOqSs4A4p5ij225Lf1tGoLRnhk2+ZjOjIGhk5rw9ERtowGZ0TGZG/V8jIdGJU1jqvPJGRtJvQKEoN4VYYhSGXyeh8ycjppsDj9FnRzrOiJH1qSuLeJqYkpiSmJKakY1CSsN55iBwi4ldgSGJIYkjaDZJu+1dXg/wEs5vKjLg7iUGJ/faofjv8EJ4kGNiRucQhkatIxsR1yBE72YlRu+C1MtgvJVzgITseslvznAmh4rrKK/QP5rCMckrjPULlWGo07U2Z79Jg4JlCkz49NHHvEkMT+42hiaHp3KHJG2NcQGYyVjM0MTS9XGj6MBi97w6O3s+UsuFeJgamk/kteypuq7Tu3mNzi4NzyBG1Y3NOGyetDsXPPcbmLI/NnRP0pDtghXp8J4ZohIGYTdQjN0CPiRB1otBaCf0SmKeYzlzJ4UkxzxpkPMvhurOEJX1qWOLepWcFS7/NwVmQ5gCN/oam9bx7d/cZ4gb82YWCqXgxfA+/4VcPfoWLYeFvLPPFsJLFWyjCV9l7+O5dQIjBu6uowdtKSSgDzGdR7uX6UiF2cvHBPXwsZhIiVpHJhj1mNEmKr4xN541NoSM8gHKAuEJTmsxe2BQMBmGK1geHGaMdAYsxYNInhpHKIU9RhKf8Fv0WJavRKVs9tTjslg7r4jCUyNd1gdBJFUSM5Vk37zjBmqW/EGIScciAKAX8pcuSFexDZ3Ugpokai7PabsuIRjVW8J48BKQlO4eANPk4jHUmrKTl7vJMWh5Lnul/R7Os18UAkhCo5CaNN7hI0CRFE2oy4uV1VZSRH5uh1soqh6S2uizsvIVmBshQ+EOKNdRYSnwxvBv3h9Mvqmm+JNsbaIo9e1LPflF1Zfb2LbTal3NCRBYmx1VIGFp1K5XFmtG6sAplxFvrKgVOGudDiBEe5BYD3s4qBa4RkpWE84KZDNx5kvG7sI5kqqO0skZCsKKerFq5JyNtsJAKIr8MEUPwJiaD+CeQjsAM3EBFP1Tq2kp4Q5G70qtF4d6u9mwRBdnmvVuxMMK9W/W9W1Xkw7ZbNrAJcv3CX4sRu9JhlSOls5Rc8Qe0wyp8Uj+btHhvVHrbVrx4ltSlT01dmqmLqYs9y9TF1PUMqSsI1NjURF0Uuxm7GLsYuxbYtZP0U3vsair9xMGZsYs923LGlRHv2hITPjxX1BBMvRrCnJZCEF4ffcYVE9OpiIlugTU5BIMxWhohcejQbOClKIwwAD4mQjRgXnoJk7CKRn2imglnyVz61MzFXV3MXOzZ/WdjMU8xTzXmKQGUoaQMJC9lGaieIVA9AxGp5wtce4hStUOudqJUHJoZuk7s2aexyuFsXHuMAcgYpLMQP9I45D7jj+c5D788cwT4W7mk+Uz8Rx1/NAg/JpAEu1Y8/PgMdRx2G35MKxKsFWJpXUKLEclNKxKS75LlYl2CxT+F8HGekgYvqVhnSmr69KTG3WNMaudCasxqzGrMaieeoQ9BAII5sZo3zGobWa0QjmJWe7mstrMAVztSayPAxZz2XDltod4AYXXSh5bKJqPBbFpTqtLpb7O/lX6X4qeKjoQReMUvsu9H4+nSbdIHB0164E1sK/AJlrM415lA4i++LC6cvZ+O4RaiiwdodjItW/N6PLqt3EgXw9vu3xctVFj729fypxpTk1sISFtt9YfrtkSdqbJKaybG+W0XbjBAi7mhomG+zipFhRfzvJL54g6erkzXX/PzagZ4dXL2xaul2/ni1eu23LY4+IDgxT76FvFMB1XPENbEiRQuYp3ChfcR/pso6zHNQQAVBhhJSqtaiFtohYIaTqHdjTigvfahCOBWhAdgbokDIlLJmhhFQW3KY3KaQI4kU0FGjaXSViNXyOj1ckGkgI8oaJDoAjLRSpbI6T2JVxAHWWGWkLEQv8C0FbZJOXqHP7HYymq9ZlYZp7fbhULZGru2aKpaq0WSOnNGG+2WGYxOGvzshrPvy3PpgBZhnrho1YKHfVyeob+JzKLD+6FiIpGZMZs4WSpp/FkCmj41oHFH2gsDtLplfOtz1DsZlvBHcDzcKujZTTdC85sADf6VGm/fGwqLvLxScZ4B1G6cX4/GECSnzSp6NLSRQZsAH7/hQaWkFHtJeXmeP3ZW88c2kE6IwkFQSxPItNmPd+oXqLWZAF7M0jJeEtZYkkvFgEndUu37uvbt5ioLYmgK2Ul72SqL+1p0Ba1z36IaFhxdFncVVHb11AEnp50J3Ui/u0yX9MeS6TqiNGalU4PuIkLgaqeGNivimIvuAQhm5QtZfaF+2qqbWQDAyqf27OfqJf+8eLVYyb/EPK+zm0at8QD3nKQdTq90yu48sjs3odd85K+CXnptW+joXsT2OmfYo3QazVSt6kb/PHzSj0DUtCehsLUCXdZJI1BRIgrvpT1PzdT13insxJE1tLVERVic+t6pUCDFqlTqcoeOEwXsVA67+sO+yOkhNVXsb4mkSrqtZaRZtQRNQx0yS9IMSxqpWHNpQ3jARw+JpT5hF5wlyulTo5xmlGOUY3cyyjHKPSeU895J76zBHjOvmOSY5JjkTkRyO8mJtSe5pnJiHPqZ5J6cO1OWz8ORP7WFOLe2q5GrFdKwxgVvdyc5G3hW/lmxnKvZych2rFXSWOfSBpBmE8s54LggnI6OWe5sWC6VmRmuYdOfJcjpU4Mcd8kxyDHIMcgxyD0zkMNgH6NwCHJKMcgxyDHIHRvk9tA+a4dy7bTPGOYY5h7XnclniwuXnbPnio/X2edipcv+Sz+e8ECr6AQFzzu1B9spzWz3BAZcgxBaOx89D7Uen+rSMsSgcZFTzRZEh9pTfEl4Aytt0obse+8wzkOthwY7fXqw4146BjsGOwY7BrvnPZMuahGlkjZuXnzKfMd8x3x3YL7bWV6tHd21kVdjtnumbDfpY6jI/kyFaCKstrlsewmrfduH+yaJqi1U0PrDJQEzPLVFUO0v+e3oU77dyEIVbdCfTBdFGdO1X1RyaFyurPuh2x8epnRkqmUZaxXWsgvwfDL6sE5uHQzKdaHcelGS/VBQEoWw5lojGPxankaKpE4hV8qO9AFYMNFgpIH3Ohy0UUoAGA8w0QIHjSJtEi2U30GB7TAscSgFNjga5xIe2i8IqU7ZTNnYTIBNCh90A7MLATXwg1guqQmIl0rrRaYutiu0NccrejK+SwUekmdLFOxFmdNCVPcgEilniY761OjI3YIvAR0bqL1pkzYx+GGUwX3Wmw2603xNPO11dp8vCQJejwaD0T3Wm4IS1VF2sl9fXS0BZVlBvGFUgHxUJ/vjbDDt3w0+U0KoJLzM3n+GcnQaKLiVqUZwZa3ObcloYOK2O914c26pUG90teazlXsMPdT63voTeeVN8Vqbo0EePMD3xDzWm2sIeXA7Ph7kqY5XUginNe2DgPt97wF5e+jNFUhW7DUAnxIIB4TxdSKuyHTYubWy4YLyKJEIUd6UMCmL/QmUi7JGL24bRlb04gqWdNia1A8GL7wUds5PUpAgrV6pDO1B8KDI2/olqfbGi7pL5rCkrZNtt3LwgZRyN7adEtT6m0C8bkuDtc65A6//tQL7JK1cdI3qGOJqnk9T9e7D5DZ8fDMdd/vDN9YY9YDQ3WryFYCj07sr2mGcn46mEAVL8LjO8Fk2/ADwcjsqRFdnH2eDfJBPXmcfZn3AjUkKzJ/6o0F/OMHwakR5WSdbicSbzE9vIMx+O+6iGu+/wxP2tg+h/aZbmvsqM/Dt6Pfbqv2aYL8wXhbpAeNLtkbYIUadNnCyaPyWYZ7qeFnWLO1A0PTYdslddk4r5yC9AYmSf/6nyI4K+HWxK0T9wa1ivjVIRaDTYH8pH7zQi/88mHqC/rNjSvmSZlyxwZSvG001UbpofZKYo56KdbJyQXgjtRRSRocBeitZQaAW864U2l9AkCg+jcolUjDU5yEl4oYzqmClaCiSl1RjHEXbNOboV/Vpk1Ha1ckq7yvosjF/PGcJ+Wk8EQO2RyRaN14RyI3VV1iD5XPrwrkpt3qoWpSpWrk2JatYL+nTIacYwjLaemI7dwIfi/mgNLGgiUKXRVmvDm1dsL4jxD5Xngu76KOxi2Z2YXZhdmF2YXY5KrtYEzWO+VGnUGB0YXR57uiyWcpsN3RpqlnG0ZHR5dyc8ysy6pZsfj9Kxpb3zMILXKctd9g1sQhbKxaBO+FU/+8OHjrwrKOm5OFOAh62RjfCdGIQUWrrNU45MmoTeEQVBG6KtXUGOoPHscHDifrcsWSVnI2RrpLx0u7hC6NpavlBx3KO/ftcoEUfDVq4v4Wh5Wk5J7vqf+pf5Vc4HwgeLfc3/d4NzfphSnlWlKIfk1KEgFjqtUNKEZop5Ql0j0jvafrPYpZOTNmEqNZm4jwtDHlS2PKQSFV7cGmnRsXowuiyk3NqHJPt5xXsZzlLzxxlDAlHCRb/9R6DSIoHkZ7EIJIT1gblaSvrerWoFzyI1JyL6oqyJxYtd9wcZAzpganDVqDOg9aL1WPLm2IfMg1lfz6Qo48IOdw/w5DDkMOQw5DzWJATrJcmFKvkhWHKYcp5KZSzXbaoPeO00SdiwmHC2Wx+tQFrGm9HR3zKa4z9bljoKS0kI9c9k4ETkzMmTbzwaT7Dd/nsr4CnypXjRZovd2KaxcHta8HVwZgGheh4eKsh1JyEadI9sLooXHekEtYr4y3O/5Wbum4CThC22oYQUD/naVIN5b/CElun2O4zQkW6QVutOyPnskjLOo3rqGKIRSQ5aC8kIQetT9ylJiuuNOiQxZWLhd5QG79ydY2O5QlG084FhfTRUIg7exiFzh2FjuWJjTi0gYbQ5I83+ZCAbDwbpvJgbq+z/jS7h/CYJS+B+WeDTtwddLLuoFp0ih14TItoAvzCCcyWyelk5PRoHToN6GmlJ6otPRmiQJPYif6knrRUdAJBrF4gCUbSS1yU8dGoSLvYRs4Gkh9ezmZV+S1pMZz0q4WonjxtqTbqxomanSLE2k4RtQEv6uWv3QMez4RtHPBk/pXE5ZtHD3l0F6zvGhG8txIe2qQS7GtDntcK3qnWehkgKuKjvJ2E3LI6G38/ge+dP++fS8TSR4tYmiMWRyyOWI8WsYIE0tZWCdrmyHuOWPz9pCNWQ+2KxhGrjXYFR6wDRSy3FrEcR6wXF7FcTcSyHetjcN5rlEVA8eP6eOUhiXcmRIxCHK44XJ13uNJHC1f8AYvDFYerxwtXTjmnlJKG+gQDxyv+fqrxqtVy9UYRq/1ydY5Zj9I5mDZf3DFYiUPO1yiSPdtQRd1rj9UX6IOBxwRJuTivuC+Qvx/4Pp94pI8Yj/gzFMcjjkenn00RjbNSeBqb8pbHpvj7rONRizWgjaJR2zWg5x2LeiN4ZvamT6NHb3Fw+6z2uM/GsJE78c6uEy85fnUVoOwoIYPQWke8m6LZEIpilEoG61xI8923hSKjo+NQxP14jxqn9NHilH4+cco+gzgl3dLXHruZC45aTaOWfeyoJb3V1jvraXZfPFrUosjFEYGj1pGilleuzQIqSH74BVRjfKLgUmBctNu9xacyvvqmO+6+zn7fHebTtFD59/l4XFkS/Ma+cZ3sd9eV5cmLi29Hw/wznllc8K+4aqTXHfRmg+40rRAejCYTyifZx6zSot3b7ke45joZ6I1u3/eHdEV/XFoeZe9nn7PRGJ/mE7p8ksOFkOY2606zIP6trBGkwzXId/AQy2n18jd/+ut3P/zmTz9+9+ZiKDtYLlxgjNXJ3sL9cDFUHfi1ckzTMfx7S4Tllqy22tYF17FGfy801N+zSkV4+vsYpLNyH/09zyG/8QfV4y24Vmahvxdq9fe8E8HhlkL4JeuiPdwOXgvpQ9Ba0Krtrb2lUqMEi057E+IqYi3L1z5EWiFNOxY6Zyurq1XwaLpmmTL2lhAv2MoeR7QjUmV5dSUTTVcGsbTo2dD6Y1VQB61qFrRyGE0Yj++ZonBUmqJ0tCRYKnyvGMyajEpLSgTBqTKvlLeLcEUwtKg4LRiHGEYZwXEr54p+utD4q7aRlAFndmltytXYEFgM7bRgcQsNK8uP+XqlLitbSK20VWXFteMtGU5AGfpolKGZMpgyTtuS4miYYeCRGZVDgXth8Cm5M2bwpNbmmCEekzNM1NrBsxh7rDRjxrPGjGeJGo+OGA1XKTZGjDarFDkwMmIcqiWp8UwnNZjrtKYMaxvuK2ml9g7FvUKIkSIK7yv5lHe/rkAG3QNr+0r64KTTXiiEDOc3UUaQzntlnIiaMeOsMCPhgk9nFipsuno2VM4u9rx+0shxLmihj4YW3HvBaHEytMAKUZPBkePRRdF5EbyE96Eze9AFD5U0pgtw6GPyRTQ2CgkRFr8C88Xz4Iu1NqtyxvL1T4szHpUrWq38bUQW7Vf+ckRkttitJb+HoDAvCNx/pLG/Y5MeYwgl4vYyXjijcJa53gc/WBm/6QCKfdQBFGMlDpsheiieqHEu6BE0blqp4CPufBehvUdQNqCK28dSqLG0mvZ8eEEfkRe4L4J54RS8cE60kKZzFqMh+8y3YFh4GrAAeBjhIU36YfV7CjAtPDYtHGa6xYuEhRbL5huhQttl8wwKDAoHHbSg37r4bYrftvjtit+++B3acsLi4PYlnzxl4jFQIZyEFNI9sLbhHkRJH3UMtP7DbJozEQLcE0Z4pZx1jApnO6ZRcyZsPBM3naE/6s/IjWfUxjP6XFBBHw0VuE+BUeGoLZl27SWbl6kObzNEgbrjohOyX2ZrZ44GDUsTIXArzV2hwWiGhqbQAE5+PGiw0gAKBEmbFmrL0HDG0FBswWsENZzw9DF+/ll+KVJvT0qTH3QqirZ63oZLyZ/gLMxT0snkE7wBC9yA5+Dga7gpHlCpqLtkhVIoyX5KFcWu9Z/68DTOryge/des9xEC0k33NrvrDrq9PBsN0172V93PFFnh8+kP85i6i4WgIpkY59ejMTwFp3tZMzb7Ck3S2sWYuuVHyeiKQBNEZTxaNHTSk0oX/DDKJqPBpzztd18mmE0wMv/5MzxKh6+zz6MZ8MEQjiY6uB4NBqN7TNEbXeU0U+Ddu3d3lLqM0HcQVwc5lMzMgzaUegIHoMBriZZefpUt0l8MAQ+G0y+qCb6k7LZwE/v2RL6tuLJw7pq3G3oWj5Bft0Ebfuh7WAo5+DpoCytfe4h7MbQ1hjZzxEGhB6WRgU2MNTLJUXpXB21OeOmcihjXFC3T3cpsGvX+pYyobUlAEk1EqDEoeokwgpSkEFRCjGuAVnQyWO9KlNjdIOERgpkpF4Qo3JcUr63g3v4FNiHU5Li+/hWPEpAZQs55OZak0OZHgd9knaDn3IpUhnDPIc0aOb8EnEUdP0IVlTWIXc7A9RErXNRdYEZEmGmCitcWW0Aq5+IS+jnlqU6htl0X6eD2IAAMaw7dkIaav7Ce2pCOFGkSLGpLPBqMmp8AtsQTwZtzgjR9VEjTDGkMaexbhjSGtJcIad7b6IMhPTapDFMaUxpTWitK26yTsjultdFK4UjOlPZEfbu70a1Of53d5ytuHt/OBl3yb4TSGFsUpi1J2bUhShvqSMoqqZ1f/NydpCiCM0k1VJyLJ0EpG+t2UZYhOK20U5tXTDkcnJROAFdAEKLZT4xSB0WpvcvRGLJwdBgTrHCVNAG3U0hIhVYWNSrrWF8bqc96NPOcUEsfFbW4Q4xRi1FrV9TqjvvTGwiv/V62CMDk7VQCyno0njCAMYAdGMCUiriVRDQIYMxfL5O/tKF5Yd5EXVJY6nmCFrP2icLX2cHYQ2I7u+FYe8EdBjIGsqfg206GzftbZIB9TUb99D1+lMFL7Z1wi597DF5GVh1oupRQxcccvLQ+aC0VyQ4oyVPMDkd8tLTO43x+tOKdPmDOZD889iDmQ4OTWwY6l2y1HcQsktIgJpX2vGBOHxnmuHeNYY5hbg5znmGOYY5hDkEtCBG09IbWeDLLPR2WU4pZ7lxYbrtQ1W4k11asijmOOa7Ot8W1o+usKM12K/d9aLdhatzuZJoFm1E1j+trseLrYa8Pril8Tc5NKb4djbNvEHWuRvfgx25GUbN++LPwed7t3WST2d3doA9FK1sDwKKsUluaWhzcrpgBD2Qd9eKn3x2nYmCcOjOcSnfBCk9J0YGopH0waQvUuIGntFZKRSeNsuF0PJVMkbCEVFZus6qCD+XQnkKMmAfzYirVWQ+TktJHzTCpS2hTTPn3Gy9TxqGgh9OhvBp3/lEu+tWRzoRYgYS5cJB73sgCRz23tdCz2Tn1nOBLHxW+uBON4Yvhi+GL4ets4StoFZyXgRYDKKYvpi+mryPSl4mtZc1MPLys2d2g+xniygQwBaKMNC67G0GYh7g5TBFoDCHlCqNwN/vQvc1JJ7S8Di6wPqWYpCTw0PvUTylfY7LSLiRb2MVQSbaqsXMzH1lfjl+NKfZ1s0kfn3kFFRH64HXr9LQKRL/IfpNfFwKkFb5IFbgYFhVBkVV/Qcm/WRI5TcyQ6jEHp1Srt9R0v5ybwov/jLxEF8KjfDaYrhAUXfggQbF/Ht8/W3VhQ82mM77hpjNRoPrn/P8eY4aMWU0n/Bt3mm1nfN2QIa609FLgfjJCBFsrJq8hNBn8IUMw3j2kC6t8oAEra0KVCmISddUWx6hoT5UU7CHEhVIh1dFREavarxYzdJrEC5Y2N0ECilbL6sVeRVQzLmzI4urVspSXV8qSFh7SZbXyEg2EJeaFmo8VBkOisZTKES0lDKVBvDRMp5NE7dpw36bRO6ocqd2mElc4NLVFwiCBVEn8FeqKqLwkU0pY6ZfldalJKoN7UkXcsGiFDOfO1POMS5uBlgoVYrdC1Y35FTbPRYRiGXz0UcFHM/gw+LB/GHwYfB4XfDDkauEDTXy3lrmHueeFck8b8a2m3NNGfIvjKnPPif2Tb3JPKsFlqhG16PwYFqd0TGrFyomHp3LX0Yk1a0oMpna7Hrn0tc92PYqVGJpvB+xPwifk8xU+MR0VpI1BS5rLHd1GQIkWcUFIS6GUAeUYgFKxvBFQvEfNKhUD6pDWU0MlnQwknVFPKRVw0MrgryL5YpI061w1pBp9VKrh3hymGqYaphqmmoZUoyG2RuO8xeEm2nCaoYahhqGmAdS014tqgjXt9aIYbBhsGvmnWvlqoqwLXoHrf4kDAHjBH/Lpv0+qzoDEyVNPyR3HGJYKwla/9ljDr3hc6omMS4GfgwPGQEKyggemtolk1qe3hDertuNKtWm1vRA4PTmxR3UwilqjWGz/YoakzmD5/Srn6CNzDnfgMOccm3Og8iXpLBxVeAg8M5v2GXQYdF4e6BgP+BJp1zxax8SccxDOSZo9a7VH26s9RAXhOPWypt6cB+e0lRlqQjltZYaYcZ4L48xbdWcPEaVAVezcW808BOnrFpRjqsoK8vvRbHCVvZ8rAlwM7XKeO+HF4uD2BeRamlD57/eY3+sNjzU1BQx7orGmdBusEEboOKmscBoe/aht7TYQholBaogR3mtjjkcYi7B9IsKoo4CKlJ9Yo5EdBqKWrNQv98bLihXeFNx1mPep0BRYVFQMcbVAzQufxp/Ofvjo7IeX2grv7Iwj3OXyZHGkIoCTciIVmkXLVjs/oOK9ssAQ+D9nt5Br/27wGZVk6qqMaaZ1mSw5v5Ohj/oN3Q6OvMqoRNDQm+8oujsSodQBEbz3qkRUun5xo5Brfzu6z267w89lDpVmmZCNzrHoRoio7OL/Hhu1SVr5xHTTjG7Qp4+JN8ooJ+AfdaD4A+BN5aN9YgKSkKlyxlqXBQZwOIdqMhajdjD0gXNDTwElWT23AZvoIz1pv5CwjZL78JJG1ZhUkSCdbg5vC9BKXRN74Fapi1Op82qnzIK5EiGZtFTI4br8dMmSM/wm82emi3MKuPl/Wp0AaQ==', 'nb27_sha256': 'e124eb65a37d6d42050047a4ff29697b1cc17cd24683cc8209f7b64a8c112711', 'nb21_sha256': 'da8c284c72fd903220e4d59c055e0d3b7b9a2d768ac4b27a00a6665f3d7a1c88', 'nb27_zip_sha256': 'e088d91a7b9087cbbe5289d3164a370002942e4b880ba0e9d9e0288292a08ace', 'nb21_zip_sha256': 'cb0bdc1c7e2cdfc1ab7d574e8c011fa7f01e2b52fad15d12dc520eac1966cb20'}
def materialize_frozen(name,sha_field):
    data=zlib.decompress(base64.b64decode(EMBEDDED[name]))
    actual=hashlib.sha256(data).hexdigest()
    assert actual==EMBEDDED[sha_field],f'Embedded evidence checksum mismatch: {name}'
    return data
history=pd.read_csv(io.BytesIO(materialize_frozen('nb27_csv','nb27_sha256')))
nb21=[json.loads(line) for line in materialize_frozen('nb21_jsonl','nb21_sha256').decode().splitlines() if line.strip()]
assert len(history)==710 and len(nb21)==64
assert history[['cohort_key','trial_id']].duplicated().sum()==0
assert len(set(r['id'] for r in nb21))==64
assert sum(int(bool(r['task_success'])) for r in nb21)==0
assert history['reconciled_success'].astype(int).sum()==72
assert (history['role']=='secondary_new_trajectory_same_problem').sum()==30
assert (history['flagged_for_human_review'].astype(int)).sum()==159
assert ((history['outcome_provenance']=='human_adjudicated')).sum()==159
print('Historical evidence PASSED: 710 existing + 64 completed NB21 = 774 distinct trial records')

### 3. Harmonize the 64 new MDLM repair trials without modifying their original labels

In [ ]:
columns=list(history.columns)
combined_records=history.to_dict(orient='records')
nb21_rows=[]
for r in nb21:
    assert r['status']=='ok' and bool(r['locked_tokens_exact'])
    row={col:np.nan for col in columns}
    row.update(cohort_key='mdlm21_fixed_per_block',cohort='MDLM controlled denoising',model='MDLM 0.5B',
      precision='fp16',native_intervention='native token infill',decoder_policy='low-confidence',
      role='model-native',trial_id=r['id'],problem_id=r['problem_id'],dataset=r['dataset'],
      action=r['action'],task_success_reported=bool(r['task_success']),reconciled_success=bool(r['task_success']),
      outcome_provenance='automated_NB21_not_human_adjudicated',flagged_for_human_review=False,
      human_adjudicated_task_success=np.nan,
      explicit_final_numeric_correct=bool(r['numeric_success']),
      automated_evidence_based_success=bool(r['task_success']),
      automated_format_and_evidence_success=bool(r['strict_success']),
      token_preservation=r['preservation'],forward_calls_native=r['nfe'],wall_seconds=r['wall_seconds'],
      original_output=r['original_output'],repaired_output=r['repaired_output'],record_fingerprint=hashlib.sha256(json.dumps(r,sort_keys=True).encode()).hexdigest())
    nb21_rows.append(row)
full=pd.concat([history,pd.DataFrame(nb21_rows,columns=columns)],ignore_index=True)
assert full.shape[0]==774 and full[['cohort_key','trial_id']].duplicated().sum()==0
assert int(full['reconciled_success'].astype(int).sum())==72
assert int(full['flagged_for_human_review'].astype(int).sum())==159
assert full['cohort_key'].nunique()==8
full.to_csv(OUT/'historical_774_frozen_evidence.csv',index=False)
summary=full.groupby(['cohort_key','cohort','role','action'],dropna=False).agg(
 trials=('trial_id','size'),successes=('reconciled_success',lambda s:int(s.astype(int).sum())),
 distinct_problems=('problem_id','nunique'),
 mean_preservation=('token_preservation','mean'),mean_native_forward_calls=('forward_calls_native','mean'),
 mean_wall_seconds=('wall_seconds','mean')).reset_index()
summary.to_csv(OUT/'historical_cohort_action_table.csv',index=False)
print(summary[['cohort_key','action','trials','successes','distinct_problems']].to_string(index=False))

### 4. Optional Notebook 29B ZIP import — never overwrite the 774-record ledger

In [ ]:
# Upload through Colab sidebar (Files > Upload), or edit this path to a Drive location.
# Leaving this file absent is valid: the historical audit still runs and exports.
NB29_ZIP_PATH=Path('/content/RepairScopeX_NB29_Semantic_Repair_results.zip')
if not NB29_ZIP_PATH.exists():
    candidates=list(Path.cwd().glob('RepairScopeX_NB29_Semantic_Repair_results*.zip'))
    if candidates:NB29_ZIP_PATH=candidates[0]
print('Notebook 29 results:',NB29_ZIP_PATH if NB29_ZIP_PATH.exists() else 'NOT PROVIDED; remaining analyses use historical 774 only')

### 5. ZIP-integrity validation and NB29 stage counts

In [ ]:
def audit_nb29_zip(path):
    if not path.exists():return None, {'state':'ABSENT','complete':False}
    with zipfile.ZipFile(path) as z:
        assert z.testzip() is None,'Corrupt ZIP member'
        names=set(z.namelist())
        assert 'EXPORT_STATUS.json' in names, 'Required NB29 export manifest missing'
        status=json.loads(z.read('EXPORT_STATUS.json'))
        for member,expected in status.get('sha256',{}).items():
            assert member in names, f'Missing archived artifact {member}'
            assert hashlib.sha256(z.read(member)).hexdigest()==expected,f'Mismatch: {member}'
        records={}
        for stage in ('natural','calibration','test_primitives','heldout','staged_primitives','staged'):
            members=sorted(n for n in names if n.startswith('records/'+stage+'/') and n.endswith('.json'))
            records[stage]=[json.loads(z.read(n)) for n in members]
            ids=[r['trial_id'] for r in records[stage]]
            assert len(ids)==len(set(ids)),f'Duplicate {stage} trial ID'
            for r in records[stage]: assert r.get('protocol_hash')==status['protocol_hash'],'Cross-protocol contamination'
        reported=status.get('stage_counts',{})
        for stage,rows in records.items():
            if stage in reported:assert len(rows)==int(reported[stage]),f'{stage} count mismatch'
        required={'calibration':96,'heldout':360,'staged':96}
        complete=all(len(records[k])==v for k,v in required.items()) and not status.get('partial',True)
        if complete:
            assert 'artifacts/research_manifest.json' in names
            meta=json.loads(z.read('artifacts/research_manifest.json'))
            assert meta.get('status')=='COMPLETE'
            if 'protocol/frozen_cohorts.json' in names:
                cohort=json.loads(z.read('protocol/frozen_cohorts.json'))
                assert len(cohort['calibration'])==12 and len(cohort['heldout'])==24
                assert not set(cohort['calibration'])&set(cohort['heldout'])
        return records, {'state':'COMPLETE' if complete else 'PARTIAL', 'complete':complete,
          'stage_counts':{k:len(v) for k,v in records.items()},'protocol_hash':status['protocol_hash'],
          'record_files_sha_verified':len(status.get('sha256',{}))}

nb29,nb29_status=audit_nb29_zip(NB29_ZIP_PATH)
(OUT/'nb29_import_status.json').write_text(json.dumps(nb29_status,indent=2))
print(json.dumps(nb29_status,indent=2))

### 6. Held-out semantic correctness and distinct-problem paired tests (NB29 complete runs only)

In [ ]:
def exact_two_sided_sign(wins,losses):
    n=wins+losses
    if n==0:return 1.0
    tail=sum(math.comb(n,k) for k in range(min(wins,losses)+1))/2**n
    return min(1.0,2.0*tail)

def paired_binary(rows,armA,armB,field='policy'):
    bypid=defaultdict(lambda:defaultdict(list))
    for r in rows: bypid[r['problem_id']][r[field]].append(r)
    table=[]
    for pid,arms in sorted(bypid.items()):
        assert armA in arms and armB in arms, f'No pair at {pid}'
        aa=arms[armA];bb=arms[armB]
        assert {r['seed'] for r in aa}=={r['seed'] for r in bb},f'Seed mismatch at {pid}'
        a=int(any(bool(r['evaluation']['success']) for r in aa))
        b=int(any(bool(r['evaluation']['success']) for r in bb))
        table.append((pid,a,b))
    wins=sum(a>b for _,a,b in table);losses=sum(a<b for _,a,b in table)
    return {'A':armA,'B':armB,'problems':len(table),'A_coverage':sum(a for _,a,b in table),
      'B_coverage':sum(b for _,a,b in table),'A_only':wins,'B_only':losses,
      'difference_coverage':(wins-losses)/len(table),'exact_sign_two_sided_p':exact_two_sided_sign(wins,losses)},table

if not nb29_status['complete']:
    print('NB29 not complete; no new primary inferential tests or performance claims are calculated.')
    policy_summary=pd.DataFrame(); paired_df=pd.DataFrame()
else:
    rows=nb29['heldout']
    assert len(rows)==360
    assert len({r['problem_id'] for r in rows})==24
    assert set(r['policy'] for r in rows)=={'global','best_fixed','adaptive','random_scope','tail50'}
    assert {r['seed'] for r in rows}=={24001,24002,24003}
    policy_summary=pd.DataFrame([{'policy':arm,'logical_trials':len(rr),
       'successful_trials':sum(bool(r['evaluation']['success']) for r in rr),
       'distinct_problems_tested':len(set(r['problem_id'] for r in rr)),
       'distinct_problems_recovered':len(set(r['problem_id'] for r in rr if r['evaluation']['success'])),
       'mean_loop_token_passes':np.mean([r['loop_token_passes'] for r in rr]),
       'mean_wall_seconds':np.mean([r['wall_seconds'] for r in rr]),
       'strict_format_successes':sum(bool(r['evaluation']['strict_correct']) for r in rr)}
       for arm in ['adaptive','global','best_fixed','random_scope','tail50']
       for rr in [[r for r in rows if r['policy']==arm]]])
    policy_summary.to_csv(OUT/'nb29_heldout_policy_summary.csv',index=False)
    comparisons=[]
    paired_rows=[]
    for opponent in ['global','best_fixed','random_scope','tail50']:
        stats,paired=paired_binary(rows,'adaptive',opponent)
        comparisons.append(stats)
        paired_rows.extend(dict(problem_id=pid,action_A='adaptive',action_B=opponent,
          A_success=a,B_success=b) for pid,a,b in paired)
    paired_df=pd.DataFrame(comparisons)
    # Multiplicity correction: Holm stepdown for four predeclared adaptive comparisons.
    order=paired_df['exact_sign_two_sided_p'].sort_values().index.to_list()
    adjusted={};running=0.0
    for i,idx in enumerate(order):
        running=max(running,float(paired_df.loc[idx,'exact_sign_two_sided_p'])*(len(order)-i))
        adjusted[idx]=min(1.0,running)
    paired_df['holm_adjusted_p']=[adjusted[i] for i in paired_df.index]
    paired_df.to_csv(OUT/'nb29_paired_problem_tests_holm.csv',index=False)
    pd.DataFrame(paired_rows).to_csv(OUT/'nb29_paired_problem_ledger.csv',index=False)
    print(policy_summary.to_string(index=False))
    print(paired_df.to_string(index=False))

### 7. Stage reuse, actual compute, and failure-eligibility audit

In [ ]:
if nb29_status['complete']:
    assert len(nb29['calibration'])==96 and len(nb29['staged'])==96
    source={r['problem_id']:r for r in nb29['natural']}
    selected=sorted({r['problem_id'] for r in nb29['heldout']})
    assert all(source[pid]['evaluation']['eligible_semantic_failure'] for pid in selected)
    assert all(source[pid]['evaluation']['failure_type']=='definite_wrong' for pid in selected)
    for r in nb29['test_primitives']+nb29['calibration']+nb29['staged_primitives']:
        assert bool(r['exact_prefix_preserved']), 'Native preserved-prefix invariant failed'
    proposals=len(nb29['test_primitives']);evals=len(nb29['heldout'])
    assert proposals<=4*24*3 and evals==360
    details={'heldout_problems':24,'physical_heldout_proposals':proposals,'logical_policy_evaluations':evals,
      'physical_staged_proposals':len(nb29['staged_primitives']),'logical_staged_evaluations':len(nb29['staged']),
      'semantic_failure_verification':'input is model-native wrong answer; ambiguous, format-only cases excluded',
      'source_problem_ids_nonoverlapping_with_calibration':True}
    (OUT/'nb29_compute_and_source_audit.json').write_text(json.dumps(details,indent=2))
    print(json.dumps(details,indent=2))
else:print('Complete experiment required before compute / sample independence conclusions.')

### 8. Publication figures (do not merge unrelated trial success rates)

In [ ]:
# Visuals use only trials and outcomes genuinely contained in the source archives.
fig,ax=plt.subplots(figsize=(10,5))
h=summary.groupby('cohort_key',as_index=False).agg(trials=('trials','sum'),successes=('successes','sum'))
ax.bar(h['cohort_key'],h['trials'])
ax.set_ylabel('Number of recorded repair trials')
ax.set_title('Historical heterogeneous cohort sizes — descriptive inventory only')
plt.xticks(rotation=25,ha='right');plt.tight_layout()
fig.savefig(OUT/'figure01_historical_trial_inventory.png',dpi=170);plt.close(fig)
if nb29_status['complete']:
    fig,ax=plt.subplots(figsize=(9,5))
    x=np.arange(len(policy_summary))
    ax.bar(x,policy_summary['distinct_problems_recovered'])
    ax.set_xticks(x,policy_summary['policy'],rotation=16,ha='right')
    ax.set_ylabel('Distinct eligible problems recovered / 24')
    ax.set_title('NB29 semantic repair: held-out problem coverage')
    fig.tight_layout();fig.savefig(OUT/'figure02_nb29_problem_coverage.png',dpi=170);plt.close(fig)
    fig,ax=plt.subplots(figsize=(9,5))
    ax.scatter(policy_summary['mean_loop_token_passes'],policy_summary['distinct_problems_recovered'])
    for _,r in policy_summary.iterrows():
        ax.annotate(r['policy'],(r['mean_loop_token_passes'],r['distinct_problems_recovered']),xytext=(4,4),textcoords='offset points')
    ax.set_xlabel('Mean native decoder loop-token passes per logical trial')
    ax.set_ylabel('Distinct problems recovered')
    ax.set_title('Observed cost-versus-coverage; not matched-compute by design')
    fig.tight_layout();fig.savefig(OUT/'figure03_nb29_compute_vs_coverage.png',dpi=170);plt.close(fig)
print('Figures saved to',OUT)

### 9. Claim audit, reporting gate, and export

In [ ]:
n29=bool(nb29_status['complete'])
claim={'historical_trials':774,'historical_cohorts':8,'historical_successes':72,
 'historical_human_adjudicated':159,'historical_automated_labels':615,
 'nb29_complete':n29,'nb29_heldout_problems':24 if n29 else None,
 'nb29_heldout_trials':360 if n29 else None,
 'allowed_claims':['NB21 completed 64/64 with zero task-successful repairs','159 flagged historical cases were adjudicated',
                  'Historical cohorts are heterogeneous and must not be pooled into one accuracy estimate'],
 'prohibited_claims':['NB29 proves superiority when p-values or paired coverage do not support this',
                      'All 774 outcomes have human review','NB29 trial evaluations are independent generations',
                      'Observed NB29 compute costs are matched across policies without a measured matching analysis']}
if n29:
    a=policy_summary.set_index('policy')
    claim['adaptive_vs_global_observed']={
      'adaptive_recovered_problems':int(a.loc['adaptive','distinct_problems_recovered']),
      'global_recovered_problems':int(a.loc['global','distinct_problems_recovered']),
      'mean_ltp_adaptive':float(a.loc['adaptive','mean_loop_token_passes']),
      'mean_ltp_global':float(a.loc['global','mean_loop_token_passes'])}
(OUT/'PUBLICATION_CLAIM_GATE.json').write_text(json.dumps(claim,indent=2)+'\n')
(OUT/'README_INTERPRETATION.md').write_text(
 '# RepairScope-X evidence audit\n\n'
 'Historically observed evidence: 710 partially reconciled + 64 MDLM controlled-denoising trials = 774.\n'
 '159 of 774 records were human adjudicated; 615 retain automated labels.\n'
 'NB29 status: '+nb29_status['state']+'. It is a separate semantic-failure benchmark.\n'
 'Never pool heterogeneous model cohort success rates, repeat seeds, or reused policy candidate records.\n'
 'Compute comparisons report observed actual LTP, not assumed budget matching.\n')
source_manifest={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(OUT.iterdir()) if p.is_file() and p.name!='SHA256_MANIFEST.json'}
(OUT/'SHA256_MANIFEST.json').write_text(json.dumps(source_manifest,indent=2)+'\n')
zip_path=Path.cwd()/'RepairScopeX_NB30_Validated_Integration_results.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED,compresslevel=8) as z:
    for p in sorted(OUT.iterdir()):
        if p.is_file():z.write(p,p.name)
with zipfile.ZipFile(zip_path) as z:
    assert z.testzip() is None
    for name,h in source_manifest.items():assert hashlib.sha256(z.read(name)).hexdigest()==h
print('Verified results ZIP:',zip_path)
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    pass